# Introdução 

1. O dataset escolhido para o trabalho foi o Global Nike Catalogue 2026. Ele foi retirado do Kaggle e possui informações de produtos da nike no total de 45 países, contendo preços, categorias, % de descontos, etc.

2. A análise desse dataset foi focada nos preços dos produtos. Pois, é um tópico muito interessante para fazer analises preditivas sobre faturamento de acordo com o poder de compra de cada país e os preços de seus produtos quando estão alinhados junto com um dataset de faturamento global da marca.

3. Nós acreditamos que haverá poucos outliers, já que pela vista inicial que demos no dataset, são muitos países desenvolvidos com moedas fortes e dados bem concentrados.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Análise Exploratória de Dados (AED)

In [ ]:
df = pd.read_csv('Global_Nike.csv')

### Primeiras linhas do dataset

In [ ]:
df.head()

### Informações gerais: tipos de dados e valores não nulos

In [ ]:
df.info()

### Dimensões do dataset (linhas x colunas)

In [ ]:
print(f'O dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas.')

### Estatísticas descritivas das colunas numéricas

Abaixo verificamos medidas como média, desvio padrão, mínimo, máximo e quartis para as variáveis numéricas do dataset.

In [ ]:
df.describe()

### Estatísticas descritivas das colunas categóricas

Verificamos quantos valores únicos existem por coluna categórica e quais são os mais frequentes.

In [ ]:
df.describe(include='object')

### Valores únicos por coluna categórica

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f'{col}: {df[col].nunique()} valores únicos → {df[col].unique()[:8]}')
    print()

### Verificação de valores ausentes

Identificamos a quantidade e o percentual de nulos em cada coluna.

In [ ]:
nulos = df.isna().sum()
pct_nulos = (nulos / len(df) * 100).round(2)
resumo_nulos = pd.DataFrame({'Nulos': nulos, '% Nulos': pct_nulos})
resumo_nulos[resumo_nulos['Nulos'] > 0]

### Distribuição das variáveis numéricas

Histogramas para entender a distribuição de preços e descontos.

In [ ]:
num_cols = df.select_dtypes(include='number').columns
df[num_cols].hist(figsize=(14, 6), bins=40, edgecolor='black', color='steelblue')
plt.suptitle('Distribuição das variáveis numéricas', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Boxplot de preço local por categoria

Identificamos visualmente a dispersão de preços e a presença de outliers por categoria de produto.

In [ ]:
df_box = df[~df['category'].isin(['PHYSICAL_GIFT_CARD', 'DIGITAL_GIFT_CARD'])]

plt.figure(figsize=(14, 6))
sns.boxplot(data=df_box, x='category', y='price_local', palette='Blues')
plt.title('Distribuição de preço local por categoria')
plt.xlabel('Categoria')
plt.ylabel('Preço local')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Distribuição de produtos por país

Verificamos quantos registros existem por país para entender o balanceamento do dataset.

In [ ]:
contagem_pais = df['country_code'].value_counts().reset_index()
contagem_pais.columns = ['country_code', 'count']

plt.figure(figsize=(20, 6))
sns.barplot(data=contagem_pais, x='country_code', y='count', color='steelblue')
plt.title('Quantidade de produtos por país')
plt.xlabel('País')
plt.ylabel('Quantidade')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Correlação entre variáveis numéricas

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='Blues')
plt.title('Matriz de correlação')
plt.tight_layout()
plt.show()

## Tratamento de Dados

Com base na AED, identificamos os seguintes problemas e aplicamos os tratamentos abaixo.

### 1. Remoção da coluna `employee_price`

A coluna `employee_price` possui uma alta taxa de valores nulos e não é relevante para a análise de preços ao consumidor final. Por isso, optamos por removê-la.

In [ ]:
df = df.drop(columns='employee_price')
print('Coluna employee_price removida.')
print(f'Colunas restantes: {list(df.columns)}')

### 2. Preenchimento de valores nulos em `discount_pct`

Os valores nulos em `discount_pct` indicam que o produto não possui desconto, e não que o dado está faltando de forma aleatória. Portanto, preenchemos com 0, o que é semanticamente correto.

In [ ]:
df['discount_pct'] = df['discount_pct'].fillna(0)
print(f'Nulos em discount_pct após tratamento: {df["discount_pct"].isna().sum()}')

### 3. Verificação e remoção de duplicatas

Verificamos se existem linhas completamente duplicadas no dataset.

In [ ]:
n_duplicados = df.duplicated().sum()
print(f'Linhas duplicadas encontradas: {n_duplicados}')

if n_duplicados > 0:
    df = df.drop_duplicates()
    print(f'Duplicatas removidas. Novo total de linhas: {len(df)}')
else:
    print('Nenhuma duplicata encontrada. Nenhuma ação necessária.')

### 4. Criação da coluna `price_usd` — conversão de moedas

Para comparar preços entre países de forma justa, convertemos `price_local` para dólares americanos (USD) usando taxas de câmbio aproximadas. Essa padronização é essencial para as visualizações do dashboard.

In [ ]:
taxa_usd = {
    'USD': 1.0,    'EUR': 1.08,   'GBP': 1.27,   'CAD': 0.74,
    'AUD': 0.63,   'NZD': 0.58,   'CHF': 1.13,   'DKK': 0.145,
    'NOK': 0.09,   'SEK': 0.09,   'CZK': 0.044,  'PLN': 0.25,
    'HUF': 0.0027, 'RON': 0.22,   'BGN': 0.55,   'HRK': 0.14,
    'ILS': 0.27,   'TRY': 0.028,  'ZAR': 0.054,  'MXN': 0.051,
    'BRL': 0.18,   'CLP': 0.001,  'COP': 0.00024,'ARS': 0.001,
    'INR': 0.012,  'IDR': 0.000062,'MYR': 0.22,  'PHP': 0.017,
    'SGD': 0.74,   'THB': 0.028,  'TWD': 0.031,  'VND': 0.000039,
    'KRW': 0.00071,'JPY': 0.0066, 'CNY': 0.138,  'EGP': 0.020,
    'HKD': 0.13,   'SAR': 0.27,   'AED': 0.27
}

df['price_usd'] = df.apply(
    lambda row: row['price_local'] * taxa_usd.get(row['currency'], None), axis=1
)

print(f'Nulos em price_usd (moedas não mapeadas): {df["price_usd"].isnull().sum()}')
print(df.groupby(['country_code', 'currency'])['price_usd'].median().reset_index())

### 5. Análise de outliers em `price_usd`

Utilizamos o método IQR (Intervalo Interquartil) para identificar outliers no preço em USD. Optamos por **manter** os outliers, pois eles representam produtos reais com preços extremos (ex.: equipamentos premium ou categorias específicas), e removê-los distorceria a análise real de precificação da Nike por país.

In [ ]:
Q1 = df['price_usd'].quantile(0.25)
Q3 = df['price_usd'].quantile(0.75)
IQR = Q3 - Q1
limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

outliers = df[(df['price_usd'] < limite_inf) | (df['price_usd'] > limite_sup)]

print(f'Q1: {Q1:.2f} | Q3: {Q3:.2f} | IQR: {IQR:.2f}')
print(f'Limite inferior: {limite_inf:.2f} | Limite superior: {limite_sup:.2f}')
print(f'\nTotal de outliers identificados: {len(outliers)} ({len(outliers)/len(df)*100:.1f}% do dataset)')
print('\nOutliers por país:')
print(outliers.groupby('country_code')['price_usd'].agg(['count', 'min', 'max']))

### 6. Verificação final dos dados tratados

In [ ]:
print('=== Dataset após tratamento ===')
print(f'Shape: {df.shape}')
print(f'\nNulos restantes:')
print(df.isna().sum()[df.isna().sum() > 0])
print('\nEstatísticas de price_usd:')
print(df['price_usd'].describe())

## Visualizações

## Gráfico 1 - Preço mediano geral em dólar/país

In [ ]:
usdp_med = df.groupby('country_code')['price_usd'].median().reset_index().sort_values(by='price_usd', ascending=False)

plt.figure(figsize=(20, 8))
plt.title('Preço mediano em dólar por país')
plt.xlabel('País')
plt.ylabel('Preço em USD')
plt.xticks(rotation=45)
sns.barplot(data=usdp_med, x='country_code', y='price_usd')
plt.tight_layout()
plt.show()

## Gráfico 2 - Preço mediano em dólar de categorias/país

In [ ]:
df_filtrado1 = df[~df['category'].isin(['PHYSICAL_GIFT_CARD', 'DIGITAL_GIFT_CARD'])]

usdp_cat_med = df_filtrado1.groupby(['country_code', 'category'])['price_usd'].median().sort_values(ascending=False).reset_index()

table1 = usdp_cat_med.pivot(index='country_code', columns='category', values='price_usd')

plt.figure(figsize=(12, 10))
sns.heatmap(data=table1, annot=True, fmt='.0f', cmap='Blues')
plt.title('Preço mediano por categoria e país')
plt.tight_layout()
plt.show()

## Gráfico 3 - % mediana de desconto/país

In [ ]:
usd_disc_med = df[df['discount_pct'] > 0].groupby('country_code')['discount_pct'].median().sort_values(ascending=False).reset_index()

plt.figure(figsize=(20, 8))
plt.title('% mediana de desconto por país')
plt.xlabel('País')
plt.ylabel('% de desconto')
plt.xticks(rotation=45)
sns.barplot(data=usd_disc_med, x='country_code', y='discount_pct')
plt.tight_layout()
plt.show()

## Gráfico 4 - % mediana de desconto de categoria/país

In [ ]:
df_filtrado2 = df[df['discount_pct'] > 0]

disc_cat_country = df_filtrado2.groupby(['country_code', 'category'])['discount_pct'].median().reset_index()

table2 = disc_cat_country.pivot(index='country_code', columns='category', values='discount_pct')

plt.figure(figsize=(12, 10))
sns.heatmap(data=table2, annot=True, fmt='.0f', cmap='Blues')
plt.title('% de desconto mediana por categoria e país')
plt.tight_layout()
plt.show()

## Gráfico 5 - Desconto vs Preço

In [ ]:
resumo = df[df['discount_pct'] > 0].groupby('country_code').agg(
    preco=('price_usd', 'median'),
    desconto=('discount_pct', 'median'),
    volume=('price_usd', 'count')
).reset_index()

plt.figure(figsize=(16, 8))
plt.title('Posicionamento de preço vs desconto por país')
plt.xlabel('Preço mediano (USD)')
plt.ylabel('Desconto mediano (%)')

plt.scatter(data=resumo, x='preco', y='desconto', s=resumo['volume']/50, alpha=0.85)

for i, row in resumo.iterrows():
    plt.annotate(row['country_code'], (row['preco'], row['desconto']), fontsize=7)

plt.tight_layout()
plt.show()

# Conclusão

1. A Nike mantém preços consistentes globalmente, com a maioria dos países entre $80-$100 USD, o que mostra uma estratégia de preço de marca consistente entre mercados.

2. Países que são outliers: Índia, Japão, Coréia do Sul e Egito. Devido a questões de posicionamento estratégico da marca ou somente por baixos descontos.

3. Nossa hipótese foi parcialmente confirmada. Apesar dos dados concentrados, encontramos outliers relevantes como a Índia $167 e o Egito $16, que mostram estratégias bem distintas dos demais países.